# Capítulo 2 - Exercício 4: Criar uma característica com `KNeighborsRegressor`

Neste exercício criamos um transformador personalizado que treina um regressor em seu método `fit()` e devolve as predições desse regressor em `transform()`. A nova característica será baseada em `latitude` e `longitude`, representando uma estimativa do preço médio dos imóveis em regiões próximas.

## Plano do exercício

1. Carregar os dados e recriar a divisão estratificada do capítulo.
2. Recriar o pré-processamento base usado no projeto.
3. Obter uma configuração razoável para o `SVR`, como no exercício anterior.
4. Criar o transformador `FeatureFromRegressor`.
5. Usar `KNeighborsRegressor` dentro desse transformador.
6. Substituir a etapa geográfica do `ColumnTransformer` por essa nova característica.
7. Avaliar o pipeline completo com validação cruzada e no conjunto de teste.

## Configuração

Importamos as bibliotecas usadas no exercício e fixamos a semente aleatória para tornar os resultados mais reprodutíveis.

In [1]:
import sys
from pathlib import Path
import tarfile
import urllib.request

import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

## Carregando os dados

Esta função segue a mesma lógica do notebook principal: baixa o arquivo caso ele ainda não exista, extrai o CSV e carrega os dados em um `DataFrame`.

In [2]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
        housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

/tmp/ipykernel_14132/2931591054.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Divisão treino/teste estratificada

Recriamos a separação do capítulo usando categorias de renda. Depois removemos `income_cat`, pois essa coluna só serve para garantir uma amostragem mais equilibrada.

In [3]:
from sklearn.model_selection import train_test_split

housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing,
    test_size=0.2,
    stratify=housing["income_cat"],
    random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

## Pré-processamento base

Começamos com o mesmo pré-processamento do projeto: razões entre atributos, logaritmo para variáveis com cauda longa, categorias codificadas e uma etapa geográfica baseada em similaridade com clusters. Mais adiante vamos substituir apenas a etapa `geo` pelo novo transformador com vizinhos mais próximos.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Similaridade com o cluster {i}" for i in range(self.n_clusters)]


log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                           "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

## Métrica de avaliação

Usaremos RMSE, a mesma métrica do projeto principal. O bloco abaixo funciona tanto em versões novas quanto antigas do Scikit-Learn.

In [5]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

## Primeiro passo: obter bons parâmetros para o `SVR`

O exercício 4 troca a engenharia de atributos geográficos, mas ainda precisa de um regressor final. Para deixar este notebook independente, fazemos uma busca aleatória curta para escolher `C`, `gamma` e `kernel` do `SVR`.

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon, loguniform
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

svr_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("svr", SVR()),
])

param_distribs = {
    "svr__kernel": ["linear", "rbf"],
    "svr__C": loguniform(20, 200_000),
    "svr__gamma": expon(scale=1.0),
}

svr_rnd_search = RandomizedSearchCV(
    svr_pipeline,
    param_distributions=param_distribs,
    n_iter=50,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

svr_rnd_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.6s
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.6s
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.6s
[CV] END svr__C=26290.20646430022, svr__gamma=0.9084469696321253, svr__kernel=rbf; total time=   0.8s
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.6s
[CV] END svr__C=84.14107900575871, svr__gamma=0.059838768608680676, svr__kernel=rbf; total time=   0.8s
[CV] END svr__C=26290.20646430022, svr__gamma=0.9084469696321253, svr__kernel=rbf; total time=   0.

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                           SimpleImputer(strategy='median')),
                                                                                          ('standardscaler',
                                                                                           StandardScaler())]),
                                                                transformers=[('bedrooms',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('functiontransformer',
                                                                                                FunctionTransformer(feature_names_...
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x7fd1391c0aa0>)])),
                                             ('svr', SVR())]),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'svr__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fd13b0b8320>,
                                        'svr__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fd1391c09e0>,
                                        'svr__kernel': ['linear', 'rbf']},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [7]:
svr_rnd_search.best_params_, -svr_rnd_search.best_score_

({'svr__C': np.float64(157055.10989448498),
  'svr__gamma': np.float64(0.26497040005002437),
  'svr__kernel': 'rbf'},
 np.float64(55805.37332417717))

## Transformador personalizado baseado em um regressor

A classe abaixo transforma qualquer regressor compatível com Scikit-Learn em um transformador. Durante `fit()`, ela clona e treina o regressor usando `X` e `y`. Durante `transform()`, ela usa esse regressor treinado para gerar uma nova coluna com as predições.

No nosso caso, `X` será formado por `latitude` e `longitude`, e `y` será `median_house_value`. Assim a nova coluna representa o preço estimado a partir dos imóveis próximos geograficamente.

In [8]:
from sklearn.base import MetaEstimatorMixin, clone
from sklearn.neighbors import KNeighborsRegressor
from sklearn.utils.validation import check_array, check_is_fitted


class FeatureFromRegressor(MetaEstimatorMixin, TransformerMixin, BaseEstimator):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y=None):
        X = check_array(X)
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        self.n_features_in_ = self.estimator_.n_features_in_
        if hasattr(self.estimator_, "feature_names_in_"):
            self.feature_names_in_ = self.estimator_.feature_names_in_
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        predictions = self.estimator_.predict(X)
        if predictions.ndim == 1:
            predictions = predictions.reshape(-1, 1)
        return predictions

    def get_feature_names_out(self, names=None):
        check_is_fitted(self)
        n_outputs = getattr(self.estimator_, "n_outputs_", 1)
        estimator_class_name = self.estimator_.__class__.__name__
        estimator_short_name = estimator_class_name.lower().replace("_", "")
        return [f"{estimator_short_name}_prediction_{i}"
                for i in range(n_outputs)]

## Testando o transformador isoladamente

Antes de colocá-lo no pipeline completo, treinamos o transformador apenas com `latitude` e `longitude`. A saída deve ser uma matriz com uma coluna: a predição do preço mediano feita pelo `KNeighborsRegressor`.

In [9]:
knn_reg = KNeighborsRegressor(n_neighbors=3, weights="distance")
knn_transformer = FeatureFromRegressor(knn_reg)
geo_features = housing[["latitude", "longitude"]]

knn_transformer.fit_transform(geo_features, housing_labels)[:5]

array([[412500.33333333],
       [435250.        ],
       [105100.        ],
       [ 96100.        ],
       [340666.66666667]])

In [10]:
knn_transformer.get_feature_names_out()

['kneighborsregressor_prediction_0']

## Adicionando a nova característica ao pré-processamento

O lugar certo para colocar esse transformador é na etapa `geo` do `ColumnTransformer`, porque ela já recebe exatamente as colunas `latitude` e `longitude`. Em vez de criar similaridades com clusters, essa etapa passa a gerar uma predição de preço com base nos vizinhos mais próximos.

In [11]:
transformers = [(name, clone(transformer), columns)
                for name, transformer, columns in preprocessing.transformers]
geo_index = [name for name, _, _ in transformers].index("geo")
transformers[geo_index] = ("geo", knn_transformer, ["latitude", "longitude"])

new_geo_preprocessing = ColumnTransformer(transformers)

## Pipeline final

O pipeline final usa o novo pré-processamento e reaproveita os melhores hiperparâmetros encontrados para o `SVR`. O importante é que, durante a validação cruzada, o transformador com KNN é treinado apenas com os dados de treino de cada divisão, evitando vazamento de dados.

In [12]:
new_geo_pipeline = Pipeline([
    ("preprocessing", new_geo_preprocessing),
    ("svr", SVR(
        C=svr_rnd_search.best_params_["svr__C"],
        gamma=svr_rnd_search.best_params_["svr__gamma"],
        kernel=svr_rnd_search.best_params_["svr__kernel"],
    )),
])

## Avaliando com validação cruzada

Agora avaliamos o pipeline completo em uma amostra de 5.000 instâncias, como no livro. O resultado pode ficar pior que a versão com similaridade por clusters; a vantagem aqui é entender como criar uma característica supervisionada dentro do pipeline.

In [13]:
from sklearn.model_selection import cross_val_score

new_pipe_rmses = -cross_val_score(
    new_geo_pipeline,
    housing.iloc[:5000],
    housing_labels.iloc[:5000],
    scoring="neg_root_mean_squared_error",
    cv=3,
)

pd.Series(new_pipe_rmses).describe()

count         3.000000
mean     104866.322819
std        2966.688335
min      101535.315061
25%      103687.330297
50%      105839.345534
75%      106531.826698
max      107224.307862
dtype: float64

## Avaliação no conjunto de teste

Por fim treinamos o pipeline na mesma amostra usada na busca e avaliamos no conjunto de teste. Em um projeto real, depois de escolher a abordagem, você poderia treinar com todo o conjunto de treino se o tempo de execução permitir.

In [14]:
new_geo_pipeline.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

new_geo_predictions = new_geo_pipeline.predict(X_test)
new_geo_rmse = root_mean_squared_error(y_test, new_geo_predictions)

new_geo_rmse

np.float64(101972.87452897223)